# Day 55 · Exercise 1: Password Hashing

**What you'll build:** Implement `hash_password(password)` and `verify_password(plain, hashed)` using `passlib` + `bcrypt`. These are the foundation of user auth — a server that stores plaintext passwords is a disaster waiting to happen.

## Setup (provided)

In [ ]:
import bcrypt as _bcrypt_lib


## Your Implementation

In [ ]:
def hash_password(password: str) -> str:
    """Hash a plaintext password using bcrypt.

    Args:
        password: The plaintext password to hash.
    Returns:
        A bcrypt hash string (starts with '$2b$').
    """
    # TODO: encode password to bytes, call _bcrypt_lib.hashpw with a new gensalt,
    #       then decode the result to a str and return it
    raise NotImplementedError

def verify_password(plain_password: str, hashed_password: str) -> bool:
    """Verify a plaintext password against a bcrypt hash.

    Args:
        plain_password:  The candidate plaintext password.
        hashed_password: The stored bcrypt hash string.
    Returns:
        True if the password matches, False otherwise.
    """
    # TODO: call _bcrypt_lib.checkpw(plain_password.encode(), hashed_password.encode())
    raise NotImplementedError


In [ ]:
def hash_password(password: str) -> str:
    return _bcrypt_lib.hashpw(password.encode(), _bcrypt_lib.gensalt()).decode()

def verify_password(plain_password: str, hashed_password: str) -> bool:
    return _bcrypt_lib.checkpw(plain_password.encode(), hashed_password.encode())


## Check Your Work

In [ ]:
def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    try:
        h = hash_password("secret123")
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: hash_password not implemented")
        print(f"\nScore: 0 / {total}")
        return

    _chk(1, isinstance(h, str) and h.startswith("$2b$"),
         f"hash_password returns a bcrypt string (got: {h[:12]}...)")

    try:
        ok_verify = verify_password("secret123", h)
        bad_verify = verify_password("wrong", h)
    except NotImplementedError:
        for i in range(2, total + 1):
            print(f"  ❌ Check {i}: verify_password not implemented")
        print(f"\nScore: {score} / {total}")
        return

    _chk(2, ok_verify is True,
         f"verify_password(correct) → True (got {ok_verify})")
    _chk(3, bad_verify is False,
         f"verify_password(wrong) → False (got {bad_verify})")

    h2 = hash_password("secret123")
    _chk(4, h != h2,
         "two hashes of the same password differ (bcrypt uses random salts)")

    _chk(5, h != "secret123",
         "hash is not the plaintext password")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Try `CryptContext(schemes=['bcrypt'], deprecated='auto', bcrypt__rounds=4)` and measure `hash_password` with `%%timeit`. Observe how halving the rounds roughly halves the time. The default of 12 rounds is chosen so each hash takes ~0.25 s — slow enough to frustrate brute-force, fast enough for login.

## Solution

<details>
<summary>Show solution</summary>

```python
def hash_password(password: str) -> str:
    return _bcrypt_lib.hashpw(password.encode(), _bcrypt_lib.gensalt()).decode()

def verify_password(plain_password: str, hashed_password: str) -> bool:
    return _bcrypt_lib.checkpw(plain_password.encode(), hashed_password.encode())
```

**Why this works:** `_bcrypt_lib.hashpw(password.encode(), _bcrypt_lib.gensalt())`
generates a fresh random salt, hashes password+salt with bcrypt, and returns the
result as bytes. Decoding to a str gives the `$2b$12$...` string you store.
`checkpw` encodes both sides and compares; it extracts the embedded salt from the
stored hash and re-derives the hash for comparison — no raw string comparison.
Two calls on the same password produce different hashes because each `gensalt()`
is unique.

</details>